# Building a Simple LLM with PyTorch: Training the Model

In this notebook, we'll train our Language Model using the preprocessed WikiText-2 dataset. We'll cover:

1. Setting up the model architecture
2. Configuring training parameters
3. Training the model
4. Monitoring training progress
5. Evaluating model performance

Let's begin!

## 1. Setup and Imports

First, let's import our dependencies and set up the environment:

In [ ]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

import torch
import matplotlib.pyplot as plt
from src.data import DataModule
from src.model import create_model
from src.trainer import create_trainer

# Set random seed for reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Prepare Data

Let's load and prepare our data using the DataModule we created:

In [ ]:
# Initialize data module
data_module = DataModule(
    seq_length=512,
    stride=256,
    batch_size=32,
    vocab_size=30000
)

# Prepare data
data_module.prepare_data()

# Get data loaders
train_dataloader = data_module.train_dataloader()
val_dataloader = data_module.val_dataloader()
test_dataloader = data_module.test_dataloader()

print(f"Vocabulary size: {data_module.vocab_size}")
print(f"Number of training batches: {len(train_dataloader)}")
print(f"Number of validation batches: {len(val_dataloader)}")
print(f"Number of test batches: {len(test_dataloader)}")

## 3. Initialize Model

Now let's create our LLM model. We'll start with a relatively small model for faster training:

In [ ]:
# Model hyperparameters
model = create_model(
    vocab_size=data_module.vocab_size,
    hidden_size=256,      # Smaller than typical models for faster training
    num_layers=6,         # Number of transformer layers
    num_heads=8          # Number of attention heads
)

# Move model to device
model = model.to(device)

# Print model summary
print("Model Architecture:")
print(f"- Vocabulary Size: {data_module.vocab_size}")
print(f"- Hidden Size: {model.config.hidden_size}")
print(f"- Number of Layers: {model.config.num_hidden_layers}")
print(f"- Number of Attention Heads: {model.config.num_attention_heads}")
print(f"- Total Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Configure Training

Let's set up our trainer with appropriate hyperparameters:

In [ ]:
# Training hyperparameters
trainer = create_trainer(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    test_dataloader=test_dataloader,
    learning_rate=3e-4,
    weight_decay=0.01,
    max_epochs=10,
    warmup_steps=1000,
    gradient_clip_val=1.0,
    checkpoint_dir="checkpoints"
)

## 5. Training Loop

Now let's train our model! This will take some time, especially if you're training on CPU:

In [ ]:
# Train the model
training_stats = trainer.train()

# Plot training curves
plt.figure(figsize=(10, 5))
plt.plot(training_stats['train_losses'], label='Training Loss')
plt.plot(training_stats['val_losses'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True)
plt.show()

print(f"Best validation loss: {training_stats['best_val_loss']:.4f}")
if training_stats['test_loss'] is not None:
    print(f"Test loss: {training_stats['test_loss']:.4f}")

## 6. Model Evaluation

Let's evaluate our model's performance by calculating perplexity and generating some sample text:

In [ ]:
from src.inference_pipeline import create_inference_pipeline

# Create inference pipeline
pipeline = create_inference_pipeline(
    model=model,
    tokenizer=data_module.tokenizer,
    device=device
)

# Test prompts
test_prompts = [
    "The history of artificial intelligence",
    "In recent scientific discoveries",
    "The impact of climate change on"
]

print("Generated Text Samples:")
print("-" * 50)

for prompt in test_prompts:
    generated_texts = pipeline.generate(
        prompt=prompt,
        max_length=100,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        num_return_sequences=1
    )
    
    print(f"Prompt: {prompt}")
    print(f"Generated: {generated_texts[0]}")
    print("-" * 50)

## 7. Save Model

Finally, let's save our trained model for later use:

In [ ]:
# Save model and tokenizer
save_dir = Path("saved_model")
save_dir.mkdir(exist_ok=True)

# Save model state
torch.save({
    'model_state_dict': model.state_dict(),
    'config': model.config.__dict__
}, save_dir / 'model.pt')

print("Model saved successfully!")

## Next Steps

We've successfully:
1. Initialized and configured our LLM model
2. Trained it on the WikiText-2 dataset
3. Evaluated its performance
4. Generated sample text

In the next notebook, we'll explore fine-tuning our model for specific tasks and improving its performance.